In [1]:
import json, os, glob
from collections import defaultdict

CASE_DIR = os.path.abspath(os.path.join("..", "cases", "murder"))
FIGURES_DIR = os.path.abspath(os.path.join("figures", "interleaved"))
os.makedirs(FIGURES_DIR, exist_ok=True)

MODE_DIRS = {"SBS": "outputs_interleaved", "EOS": "outputs_eos_interleaved"}

MODEL_NAMES = {
    "claude-3-haiku-20240307": "Claude 3 Haiku",
    "claude-3-5-haiku-20241022": "Claude 3.5 Haiku",
    "claude-3-7-sonnet-20250219": "Claude 3.7 Sonnet",
    "claude-sonnet-4-20250514": "Claude 4 Sonnet",
    "claude-sonnet-4-6": "Claude Sonnet 4.6",
    "gemini-2.0-flash": "Gemini 2.0 Flash",
    "gemini-2.5-flash": "Gemini 2.5 Flash",
    "gemini-3-flash-preview": "Gemini 3 Flash",
    "gpt-3.5-turbo": "GPT 3.5 Turbo",
    "gpt-4o": "GPT 4o",
    "gpt-5": "GPT 5",
    "gpt-5.4": "GPT 5.4",
    "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8": "LLaMA 4 Maverick",
    "Qwen/Qwen2.5-72B-Instruct-Turbo": "Qwen 2.5 72B",
}

def scrape_counts(case_dir, out_dir):
    counts = defaultdict(lambda: {"dp": [0, 0], "pd": [0, 0]})
    for order in ["dp", "pd"]:
        order_path = os.path.join(case_dir, out_dir, order)
        if not os.path.isdir(order_path):
            continue
        for path in glob.glob(os.path.join(order_path, "**", "judgments_run*.json"), recursive=True):
            rel = os.path.relpath(os.path.dirname(path), order_path)
            with open(path) as f:
                d = json.load(f)
            if not isinstance(d[-1], bool):
                continue
            if d[-1] is True:
                counts[rel][order][0] += 1
            else:
                counts[rel][order][1] += 1
    return counts

data = {}
for mode, out_dir in MODE_DIRS.items():
    for model_key, orders in scrape_counts(CASE_DIR, out_dir).items():
        display = MODEL_NAMES.get(model_key, model_key)
        if display not in data:
            data[display] = {}
        data[display][mode] = {"DP": orders["dp"], "PD": orders["pd"]}

print(f"Loaded {len(data)} models")
for m, v in sorted(data.items()):
    print(f"  {m}: {v}")

Loaded 11 models
  Claude 4 Sonnet: {'SBS': {'DP': [30, 0], 'PD': [1, 29]}, 'EOS': {'DP': [30, 0], 'PD': [0, 30]}}
  Claude Sonnet 4.6: {'SBS': {'DP': [12, 18], 'PD': [0, 30]}, 'EOS': {'DP': [30, 0], 'PD': [0, 30]}}
  GPT 3.5 Turbo: {'SBS': {'DP': [30, 0], 'PD': [14, 16]}, 'EOS': {'DP': [29, 1], 'PD': [2, 28]}}
  GPT 4o: {'SBS': {'DP': [27, 3], 'PD': [14, 16]}, 'EOS': {'DP': [30, 0], 'PD': [0, 30]}}
  GPT 5: {'SBS': {'DP': [6, 4], 'PD': [0, 0]}, 'EOS': {'DP': [13, 17], 'PD': [3, 27]}}
  GPT 5.4: {'SBS': {'DP': [26, 4], 'PD': [0, 30]}, 'EOS': {'DP': [27, 3], 'PD': [0, 30]}}
  Gemini 2.0 Flash: {'SBS': {'DP': [27, 3], 'PD': [0, 30]}, 'EOS': {'DP': [29, 1], 'PD': [0, 30]}}
  Gemini 2.5 Flash: {'SBS': {'DP': [18, 12], 'PD': [14, 16]}, 'EOS': {'DP': [28, 2], 'PD': [2, 28]}}
  Gemini 3 Flash: {'SBS': {'DP': [6, 24], 'PD': [0, 30]}, 'EOS': {'DP': [12, 18], 'PD': [0, 30]}}
  LLaMA 4 Maverick: {'SBS': {'DP': [30, 0], 'PD': [0, 30]}, 'EOS': {'DP': [30, 0], 'PD': [0, 30]}}
  Qwen 2.5 72B: {'SBS':

In [2]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch

guilty_color = "#2c2f7b"
not_guilty_color = "#a3a5d9"

def proportions(vals):
    total = sum(vals)
    return np.array(vals) / total if total > 0 else np.array([0.0, 0.0])

models = sorted(data.keys())

fig = plt.figure(figsize=(9, 2.5 * len(models)))
gs = GridSpec(
    nrows=len(models),
    ncols=3,
    width_ratios=[1.4, 3, 3],
    hspace=0.5,
    wspace=0.35,
)

for i, model in enumerate(models):
    ax_label = fig.add_subplot(gs[i, 0])
    ax_label.axis("off")
    ax_label.text(1.0, 0.5, model, ha="right", va="center", fontsize=11, fontweight="bold")

    for j, mode in enumerate(["EOS", "SBS"]):
        ax = fig.add_subplot(gs[i, j + 1])

        if mode not in data[model]:
            ax.set_title(mode, fontsize=11, fontweight="bold")
            ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes, color="gray")
            continue

        dp = proportions(data[model][mode]["DP"])
        pd = proportions(data[model][mode]["PD"])
        x = np.arange(2)
        width = 0.6

        ax.bar(x, [dp[0], pd[0]], width, color=guilty_color)
        ax.bar(x, [dp[1], pd[1]], width, bottom=[dp[0], pd[0]], color=not_guilty_color)

        ax.set_xticks(x)
        ax.set_xticklabels(["DP", "PD"], fontsize=9)
        ax.set_ylim(0, 1)
        ax.grid(axis="y", linestyle="--", alpha=0.35)
        ax.set_title(mode, fontsize=11, fontweight="bold")
        if j == 0:
            ax.set_ylabel("Proportion", fontsize=9)

legend_handles = [
    Patch(facecolor=guilty_color, label="Guilty"),
    Patch(facecolor=not_guilty_color, label="Not Guilty"),
]
fig.legend(handles=legend_handles, title="Verdict", loc="upper center", ncol=2, frameon=True, bbox_to_anchor=(0.5, 1.02))
plt.subplots_adjust(top=0.97)

filepath = os.path.join(FIGURES_DIR, "all_models_interleaved_grid.png")
plt.savefig(filepath, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved {filepath}")

Saved /Users/addisonwu/Desktop/llm_sbs_eos_positional_bias/analysis/figures/interleaved/all_models_interleaved_grid.png
